In [0]:
from pyspark.sql import functions as F
path = "/Volumes/business_to_business/sports_bar_data/customers/customers.csv"



Bronze Layer operations

In [0]:
customers_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(path)
    .withColumn("ingestion_time", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
customers_df.show()

+-----------+--------------------+----------+--------------------+-------------+---------+
|customer_id|       customer_name|      city|      ingestion_time|    file_name|file_size|
+-----------+--------------------+----------+--------------------+-------------+---------+
|     789201|      FitFuel Market| Bengaluru|2026-09-04 09:28:...|customers.csv|     1404|
|     789202|      FitFuel Market| Hyderabad|2026-09-04 09:28:...|customers.csv|     1404|
|     789203|      FitFuel Market| New Delhi|2026-09-04 09:28:...|customers.csv|     1404|
|     789301|Athlete's Choice ...| Bengaluru|2026-09-04 09:28:...|customers.csv|     1404|
|     789303|Athlete's Choice ...| New Delhi|2026-09-04 09:28:...|customers.csv|     1404|
|     789101|     Endurance Foods| Bengalore|2026-09-04 09:28:...|customers.csv|     1404|
|     789102|     Endurance Foods| Hyderabad|2026-09-04 09:28:...|customers.csv|     1404|
|     789103|     Endurance Foods| New Delhi|2026-09-04 09:28:...|customers.csv|     1404|

In [0]:
customers_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



In [0]:
customers_df.write.mode("overwrite").format("delta").saveAsTable("business_to_business.bronze.customers")

### Silver Layer

In [0]:
silver_df = (
    spark.read.table("business_to_business.bronze.customers")
)

In [0]:
silver_df.show()

+-----------+--------------------+----------+--------------------+-------------+---------+
|customer_id|       customer_name|      city|      ingestion_time|    file_name|file_size|
+-----------+--------------------+----------+--------------------+-------------+---------+
|     789201|      FitFuel Market| Bengaluru|2026-09-04 09:28:...|customers.csv|     1404|
|     789202|      FitFuel Market| Hyderabad|2026-09-04 09:28:...|customers.csv|     1404|
|     789203|      FitFuel Market| New Delhi|2026-09-04 09:28:...|customers.csv|     1404|
|     789301|Athlete's Choice ...| Bengaluru|2026-09-04 09:28:...|customers.csv|     1404|
|     789303|Athlete's Choice ...| New Delhi|2026-09-04 09:28:...|customers.csv|     1404|
|     789101|     Endurance Foods| Bengalore|2026-09-04 09:28:...|customers.csv|     1404|
|     789102|     Endurance Foods| Hyderabad|2026-09-04 09:28:...|customers.csv|     1404|
|     789103|     Endurance Foods| New Delhi|2026-09-04 09:28:...|customers.csv|     1404|

In [0]:
silver_df.withColumn("customer_name",
                     F.when(F.col("customer_name").isNull(), "Unknown")
                     .otherwise(F.trim(F.col("customer_name")))
                       )

DataFrame[customer_id: int, customer_name: string, city: string, ingestion_time: timestamp, file_name: string, file_size: bigint]

In [0]:
display(silver_df)

customer_id,customer_name,city,ingestion_time,file_name,file_size
789201,FitFuel Market,Bengaluru,2026-09-04T09:28:45.051Z,customers.csv,1404
789202,FitFuel Market,Hyderabad,2026-09-04T09:28:45.051Z,customers.csv,1404
789203,FitFuel Market,New Delhi,2026-09-04T09:28:45.051Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-09-04T09:28:45.051Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2026-09-04T09:28:45.051Z,customers.csv,1404
789101,Endurance Foods,Bengalore,2026-09-04T09:28:45.051Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-09-04T09:28:45.051Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-09-04T09:28:45.051Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2026-09-04T09:28:45.051Z,customers.csv,1404
789122,HydroBoost Nutrition,New Delhi,2026-09-04T09:28:45.051Z,customers.csv,1404


In [0]:
silver_df.filter( F.col("city").isNull()).show(truncate=False)

+-----------+---------------------+----+--------------------------+-------------+---------+
|customer_id|customer_name        |city|ingestion_time            |file_name    |file_size|
+-----------+---------------------+----+--------------------------+-------------+---------+
|789403     |SprintX Nutrition    |NULL|2026-09-04 09:28:45.051028|customers.csv|1404     |
|789420     | ZenAthlete foods    |NULL|2026-09-04 09:28:45.051028|customers.csv|1404     |
|789521     | PrimeFuel Nutrition |NULL|2026-09-04 09:28:45.051028|customers.csv|1404     |
|789603     |Recovery Lane        |NULL|2026-09-04 09:28:45.051028|customers.csv|1404     |
|789603     |Recovery Lane        |NULL|2026-09-04 09:28:45.051028|customers.csv|1404     |
+-----------+---------------------+----+--------------------------+-------------+---------+



In [0]:
silver_df.select("city").distinct().show()

+----------+
|      city|
+----------+
| Bengaluru|
| Hyderabad|
| New Delhi|
| Bengalore|
|Hyderabadd|
|  Hyderbad|
| NewDelhee|
|  NewDelhi|
|Bengaluruu|
|  NewDheli|
|      NULL|
+----------+



In [0]:
city_mapping = {
    "Bengalore" : "Bangalore",
    "Bengaluru" : "Bangalore",
    "Hyderabadd" : "Hyderabad",
    "NewDelhi"   : "New Delhi",
    "NewDheli" : "New Delhi",
    "NewDelhee": "New Delhi",
    "Bengaluruu" : "Bangalore",
    "Bengaluru" : "Bangalore",
    "Hyderbad" :"Hyderabad"
}

In [0]:
silver_df.groupBy("customer_id").agg(F.count("customer_id").alias("count")).filter(F.col("count") >= 2).show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|     789321|    2|
|     789503|    2|
|     789522|    2|
|     789603|    2|
+-----------+-----+



In [0]:
silver_df.count()

39

In [0]:
silver_df =  silver_df.dropDuplicates()

In [0]:
silver_df.count()

35

In [0]:
(
    silver_df.groupBy("customer_id")
    .agg(F.count("customer_id").alias("count"))
    .filter(F.col("count") >= 2 )
    .count()
)

0

In [0]:
allowed_cities = ["Hyderabad", "Bangalore", "New Delhi"]
silver_df = silver_df.replace(city_mapping, subset=["city"])\
    .withColumn("city",  
                F.when(F.col("city").isNull(), None)\
                .when(F.col("city").isin(allowed_cities), F.col("city") )
                .otherwise(None)
                ) 

In [0]:
silver_df.select(F.col("city")).distinct().show()

+---------+
|     city|
+---------+
|Bangalore|
|Hyderabad|
|New Delhi|
|     NULL|
+---------+



In [0]:
city_mapping

{'Bengalore': 'Bangalore',
 'Bengaluru': 'Bangalore',
 'Hyderabadd': 'Hyderabad',
 'NewDelhi': 'New Delhi',
 'NewDheli': 'New Delhi',
 'NewDelhee': 'New Delhi',
 'Bengaluruu': 'Bangalore',
 'Hyderbad': 'Hyderabad'}

In [0]:
mapping_data = [ 
                (wrong_city, correct_city) for (wrong_city, correct_city) in city_mapping.items()
]

In [0]:
mapping_data

[('Bengalore', 'Bangalore'),
 ('Bengaluru', 'Bangalore'),
 ('Hyderabadd', 'Hyderabad'),
 ('NewDelhi', 'New Delhi'),
 ('NewDheli', 'New Delhi'),
 ('NewDelhee', 'New Delhi'),
 ('Bengaluruu', 'Bangalore'),
 ('Hyderbad', 'Hyderabad')]

In [0]:
mapping_df = spark.createDataFrame(mapping_data, ["incorrect_city", "correct_city"])
mapping_df.show()

+--------------+------------+
|incorrect_city|correct_city|
+--------------+------------+
|     Bengalore|   Bangalore|
|     Bengaluru|   Bangalore|
|    Hyderabadd|   Hyderabad|
|      NewDelhi|   New Delhi|
|      NewDheli|   New Delhi|
|     NewDelhee|   New Delhi|
|    Bengaluruu|   Bangalore|
|      Hyderbad|   Hyderabad|
+--------------+------------+



In [0]:
silver_df.join(mapping_df, F.col("incorrect_city") == F.col("city"), "left")\
    .select("city", "correct_city").show()


+---------+------------+
|     city|correct_city|
+---------+------------+
|Hyderabad|        NULL|
|Bangalore|        NULL|
|     NULL|        NULL|
|Hyderabad|        NULL|
|Bangalore|        NULL|
|     NULL|        NULL|
|Bangalore|        NULL|
|Bangalore|        NULL|
|New Delhi|        NULL|
|Hyderabad|        NULL|
|New Delhi|        NULL|
|Hyderabad|        NULL|
|Bangalore|        NULL|
|New Delhi|        NULL|
|Bangalore|        NULL|
|Hyderabad|        NULL|
|Hyderabad|        NULL|
|New Delhi|        NULL|
|Bangalore|        NULL|
|Hyderabad|        NULL|
+---------+------------+
only showing top 20 rows


In [0]:
silver_df = silver_df.withColumn("customer_name", F.trim("customer_name"))
silver_df.show()

+-----------+--------------------+---------+--------------------+-------------+---------+
|customer_id|       customer_name|     city|      ingestion_time|    file_name|file_size|
+-----------+--------------------+---------+--------------------+-------------+---------+
|     789201|      FitFuel Market|Bangalore|2026-09-04 09:28:...|customers.csv|     1404|
|     789202|      FitFuel Market|Hyderabad|2026-09-04 09:28:...|customers.csv|     1404|
|     789203|      FitFuel Market|New Delhi|2026-09-04 09:28:...|customers.csv|     1404|
|     789301|Athlete's Choice ...|Bangalore|2026-09-04 09:28:...|customers.csv|     1404|
|     789303|Athlete's Choice ...|New Delhi|2026-09-04 09:28:...|customers.csv|     1404|
|     789101|     Endurance Foods|Bangalore|2026-09-04 09:28:...|customers.csv|     1404|
|     789102|     Endurance Foods|Hyderabad|2026-09-04 09:28:...|customers.csv|     1404|
|     789103|     Endurance Foods|New Delhi|2026-09-04 09:28:...|customers.csv|     1404|
|     7891

In [0]:
silver_df.select("customer_name").distinct().count()

21

In [0]:
silver_df = silver_df.withColumn("customer_name", 
                     F.when(F.col("customer_name").isNotNull(), F.initcap(F.col("customer_name")))
                        .otherwise(None)

                     )

In [0]:
silver_df.select("customer_name").show(truncate=False)

+----------------------+
|customer_name         |
+----------------------+
|Fitfuel Market        |
|Fitfuel Market        |
|Fitfuel Market        |
|Athlete's Choice Store|
|Athlete's Choice Store|
|Endurance Foods       |
|Endurance Foods       |
|Endurance Foods       |
|Hydroboost Nutrition  |
|Hydroboost Nutrition  |
|Macrobite Superfoods  |
|Macrobite Superfoods  |
|Powersnack Hub        |
|Powersnack Hub        |
|Sprintx Nutrition     |
|Sprintx Nutrition     |
|Sprintx Nutrition     |
|Zenathlete Foods      |
|Zenathlete Foods      |
|Zenathlete Foods      |
+----------------------+
only showing top 20 rows


In [0]:
null_city_customers = []
for  row in silver_df.filter(F.col("city").isNull()).collect():
    null_city_customers.append(row["customer_name"])
print(null_city_customers)


['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']


In [0]:
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}
city_data = [(customer_id, city) for customer_id, city in customer_city_fix.items()]
print(city_data)
city_df = spark.createDataFrame(city_data, ["customer_id", "fixed_city"])

silver_df = silver_df.join(city_df, on="customer_id", how="left")\
    .withColumn("city", F.coalesce(F.col("city"), F.col("fixed_city")))\
    .drop("fixed_city")\



[(789403, 'New Delhi'), (789420, 'Bengaluru'), (789521, 'Hyderabad'), (789603, 'Hyderabad')]


In [0]:
silver_df.show()

+-----------+--------------------+---------+--------------------+-------------+---------+
|customer_id|       customer_name|     city|      ingestion_time|    file_name|file_size|
+-----------+--------------------+---------+--------------------+-------------+---------+
|     789421|    Zenathlete Foods|Hyderabad|2026-09-04 09:28:...|customers.csv|     1404|
|     789301|Athlete's Choice ...|Bangalore|2026-09-04 09:28:...|customers.csv|     1404|
|     789420|    Zenathlete Foods|Bengaluru|2026-09-04 09:28:...|customers.csv|     1404|
|     789102|     Endurance Foods|Hyderabad|2026-09-04 09:28:...|customers.csv|     1404|
|     789501|Peak Performance ...|Bangalore|2026-09-04 09:28:...|customers.csv|     1404|
|     789603|       Recovery Lane|Hyderabad|2026-09-04 09:28:...|customers.csv|     1404|
|     789220|Macrobite Superfoods|Bangalore|2026-09-04 09:28:...|customers.csv|     1404|
|     789320|      Powersnack Hub|Bangalore|2026-09-04 09:28:...|customers.csv|     1404|
|     7894

In [0]:
silver_df.filter(F.col("city").isNull()).count()

0

In [0]:
# Perform sanity checks on the cleaned silver_df

# 1. Check for null values in critical columns
print("=== Null Value Check ===")
null_counts = silver_df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in silver_df.columns]
)
null_counts.show()

# 2. Check for duplicate customer_ids
print("\n=== Duplicate Customer ID Check ===")
duplicate_count = (
    silver_df.groupBy("customer_id")
    .agg(F.count("customer_id").alias("count"))
    .filter(F.col("count") >= 2)
    .count()
)
print(f"Number of duplicate customer_ids: {duplicate_count}")

# 3. Verify city values are in allowed list
print("\n=== City Values Check ===")
allowed_cities = ["Hyderabad", "Bangalore", "New Delhi"]
silver_df.select("city").distinct().show()
invalid_city_count = silver_df.filter(
    (~F.col("city").isin(allowed_cities)) & (F.col("city").isNotNull())
).count()
print(f"Number of records with invalid cities: {invalid_city_count}")

# 4. Check total record count
print("\n=== Total Record Count ===")
total_count = silver_df.count()
print(f"Total records in silver_df: {total_count}")

# 5. Display sample of cleaned data
print("\n=== Sample of Cleaned Data ===")
silver_df.show(10, truncate=False)


=== Null Value Check ===
+-----------+-------------+----+--------------+---------+---------+
|customer_id|customer_name|city|ingestion_time|file_name|file_size|
+-----------+-------------+----+--------------+---------+---------+
|          0|            0|   0|             0|        0|        0|
+-----------+-------------+----+--------------+---------+---------+


=== Duplicate Customer ID Check ===
Number of duplicate customer_ids: 0

=== City Values Check ===
+---------+
|     city|
+---------+
|Hyderabad|
|Bangalore|
|Bengaluru|
|New Delhi|
+---------+

Number of records with invalid cities: 1

=== Total Record Count ===
Total records in silver_df: 35

=== Sample of Cleaned Data ===
+-----------+----------------------+---------+--------------------------+-------------+---------+
|customer_id|customer_name         |city     |ingestion_time            |file_name    |file_size|
+-----------+----------------------+---------+--------------------------+-------------+---------+
|789421    

In [0]:
# Null checks 
silver_df.select(
    [ F.sum(F.when(F.col(c).isNull(), 1)\
        .otherwise(0)
        
            ).alias(f"{c}_count") for c in silver_df.columns ]
).show()

# duplicate customer id's
silver_df.groupBy("customer_id")\
    .agg(F.count("customer_id").alias("count"))\
        .filter(F.col("count") >= 2)\
        .show()

# verify the city values in the  allowed_list
print(f"Total not allowed citites in the data {silver_df.filter(~F.col("city").isin(allowed_cities)).select("city").count()} ")

# total records count 
print(f"Total records in the silver layer is {silver_df.count()} " )

# display 10 sample records
silver_df.show(10, truncate=False)



+-----------------+-------------------+----------+--------------------+---------------+---------------+
|customer_id_count|customer_name_count|city_count|ingestion_time_count|file_name_count|file_size_count|
+-----------------+-------------------+----------+--------------------+---------------+---------------+
|                0|                  0|         0|                   0|              0|              0|
+-----------------+-------------------+----------+--------------------+---------------+---------------+

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+

Total not allowed citites in the data 1 
Total records in the silver layer is 35 
+-----------+----------------------+---------+--------------------------+-------------+---------+
|customer_id|customer_name         |city     |ingestion_time            |file_name    |file_size|
+-----------+----------------------+---------+--------------------------+-------------+---------+
|789421     |Zenathlet

In [0]:
allowed_cities

['Hyderabad', 'Bangalore', 'New Delhi']

In [0]:
silver_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



In [0]:
silver_df = silver_df.withColumn("customer_id", F.col("customer_id").cast("string"))


In [0]:
silver_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



### Standardize the column names to match with the parent schema 

In [0]:
# gold-> customer_code	customer	market	platform	channel
# silver->customer_id,customer_name 
silver_df = (
    silver_df.withColumn("market", F.lit("India"))
    .withColumn("customer", F.concat_ws("-",F.col("customer_name"), F.coalesce(F.col("city"), F.lit("Unknown"))))
    .withColumn("platform",F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Aquistion"))
)

In [0]:
silver_df.write\
    .mode("overwrite")\
        .format("delta")\
            .option("delta.enableChangeDataFeed", "true") \
                .option("mergeSchema", "true")\
                    .saveAsTable("business_to_business.silver.dim_customer")

### capturing the required field into Gold Layer


In [0]:
gold_df = spark.sql("select * from  business_to_business.silver.dim_customer")

In [0]:
gold_df.show()


+-----------+--------------------+---------+--------------------+-------------+---------+------+--------------------+----------+---------+
|customer_id|       customer_name|     city|      ingestion_time|    file_name|file_size|market|            customer|  platform|  channel|
+-----------+--------------------+---------+--------------------+-------------+---------+------+--------------------+----------+---------+
|     789421|    Zenathlete Foods|Hyderabad|2026-09-04 09:28:...|customers.csv|     1404| India|Zenathlete Foods-...|Sports Bar|Aquistion|
|     789301|Athlete's Choice ...|Bangalore|2026-09-04 09:28:...|customers.csv|     1404| India|Athlete's Choice ...|Sports Bar|Aquistion|
|     789420|    Zenathlete Foods|Bengaluru|2026-09-04 09:28:...|customers.csv|     1404| India|Zenathlete Foods-...|Sports Bar|Aquistion|
|     789102|     Endurance Foods|Hyderabad|2026-09-04 09:28:...|customers.csv|     1404| India|Endurance Foods-H...|Sports Bar|Aquistion|
|     789501|Peak Performan

In [0]:
# taking the required fields from the dataframe
gold_df = (
    gold_df.select(
        "customer_id",
        "customer_name",
        "city",
        "market",
        "customer",
        "platform",
        "channel"
    )
)

In [0]:
gold_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- market: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)



In [0]:
gold_df.write\
    .mode("overwrite")\
.format("delta")\
        .option("mergeSchema", "true")\
.option("delta.enableChangeDataFeed", "true")\
                .saveAsTable("business_to_business.gold.dim_sports_bar_customer")

In [0]:
from delta.tables import DeltaTable 
delta_table = DeltaTable.forName(spark, "business_to_business.gold.dim_customers")
child_table = spark.sql(
    "select * from business_to_business.gold.dim_sports_bar_customer"
)\
.select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
child_table.show()

+-------------+--------------------+------+----------+---------+
|customer_code|            customer|market|  platform|  channel|
+-------------+--------------------+------+----------+---------+
|       789421|Zenathlete Foods-...| India|Sports Bar|Aquistion|
|       789301|Athlete's Choice ...| India|Sports Bar|Aquistion|
|       789420|Zenathlete Foods-...| India|Sports Bar|Aquistion|
|       789102|Endurance Foods-H...| India|Sports Bar|Aquistion|
|       789501|Peak Performance ...| India|Sports Bar|Aquistion|
|       789603|Recovery Lane-Hyd...| India|Sports Bar|Aquistion|
|       789220|Macrobite Superfo...| India|Sports Bar|Aquistion|
|       789320|Powersnack Hub-Ba...| India|Sports Bar|Aquistion|
|       789422|Zenathlete Foods-...| India|Sports Bar|Aquistion|
|       789621|Eliteathlete Nutr...| India|Sports Bar|Aquistion|
|       789122|Hydroboost Nutrit...| India|Sports Bar|Aquistion|
|       789221|Macrobite Superfo...| India|Sports Bar|Aquistion|
|       789520|Primefuel 

In [0]:
# merge operation
delta_table.alias("target").merge(
    source=child_table.alias("source"),\
    condition="target.customer_code == source.customer_code"\
)\
.whenMatchedUpdateAll()\
.whenNotMatchedInsertAll()\
.execute()


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]